In [2]:
import pandas as pd
df = pd.read_csv('docs/tracks.csv')
df.head()

/var/folders/lq/r8qjzl2x10b2vksfq2gq9m080000gn/T/ipykernel_8913/3304437656.py:2: DtypeWarning: Columns (0: explicit, 1: added_at) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('docs/tracks.csv')


,track_id,streams,artist_followers,genres,album_total_tracks,track_artists,artist_popularity,explicit,tempo,chart,...,speechiness,region,danceability,valence,acousticness,liveness,trend,instrumentalness,loudness,name
0,07vS8obfeZbr8H4MgQfXR7,NaN,2338837.0,"['indie pop', 'la indie', 'pov: indie']",2.0,Phoebe Bridgers,74.0,False,97.129,NaN,...,0.0407,NaN,0.373,0.138,0.9480,0.0816,NaN,0.000000,-15.193,Friday I’m In Love - Recorded at Spotify Studi...
1,1PEqh7awkpuepLBSq8ZwqD,27156.0,84914.0,"['lilith', 'new wave pop']",11.0,NaN,51.0,False,103.773,top200,...,0.0348,France,0.744,0.122,0.6270,0.0898,NEW_ENTRY,0.421000,-11.977,I Love You Always Forever
2,7E8pPgBY84oDaXRcqODavR,NaN,59150.0,"['deep groove house', 'house', 'tech house']",2.0,NaN,54.0,False,122.030,NaN,...,0.0357,NaN,0.747,0.897,0.0794,0.3700,NaN,0.000531,-5.209,Love Too Deep - Radio Edit
3,0Atml4huw4Fgyk6YSHiK4M,NaN,1528.0,[],15.0,NaN,0.0,False,84.099,NaN,...,0.0356,NaN,0.604,0.564,0.1000,0.0865,NaN,0.000000,-7.097,No Tiren Las Botellas
4,4WYDmIZrwxBHdBYdvi5oQO,NaN,6776.0,"['chill lounge', 'deep chill']",26.0,NaN,28.0,False,156.017,NaN,...,0.0613,NaN,0.761,0.761,0.0616,0.0822,NaN,0.873000,-10.961,El Momento de Despertar - Blue Sky Mix


In [3]:
import pandas as pd
df = pd.read_parquet('docs/tracks.parquet')
print('Total rows:', len(df))
print('Columns:', df.columns.tolist())

Total rows: 899702
Columns: ['track_id', 'streams', 'artist_followers', 'genres', 'album_total_tracks', 'track_artists', 'artist_popularity', 'explicit', 'tempo', 'chart', 'album_release_date', 'energy', 'key', 'added_at', 'popularity', 'track_album_album', 'duration_ms', 'available_markets', 'track_track_number', 'rank', 'mode', 'time_signature', 'album_name', 'speechiness', 'region', 'danceability', 'valence', 'acousticness', 'liveness', 'trend', 'instrumentalness', 'loudness', 'name']


In [4]:
import pandas as pd

df = pd.read_parquet('docs/tracks.parquet')
print('=== DATASET OVERVIEW ===')
print('Total Songs in dataset:', len(df))

print('\n=== JUSTIN BIEBER SONGS ===')
jb_mask = df['track_artists'].astype(str).str.contains('Justin Bieber', case=False, na=False) | df['name'].astype(str).str.contains('Justin Bieber', case=False, na=False)
jb_songs = df[jb_mask][['name', 'track_artists', 'album_name']].drop_duplicates(subset=['name'])
print(f'Total Justin Bieber songs found: {len(jb_songs)}')
print(jb_songs.head(15).to_string(index=False))

print('\n=== SHAWN MENDES SONGS ===')
sm_mask = df['track_artists'].astype(str).str.contains('Shawn Mendes', case=False, na=False) | df['name'].astype(str).str.contains('Shawn Mendes', case=False, na=False)
sm_songs = df[sm_mask][['name', 'track_artists', 'album_name']].drop_duplicates(subset=['name'])
print(f'Total Shawn Mendes songs found: {len(sm_songs)}')
print(sm_songs.head(15).to_string(index=False))

print('\n=== REGION / MARKET DISTRIBUTION ===')
if 'region' in df.columns:
    print('Region counts (Top 20):')
    print(df['region'].value_counts(dropna=False).head(20))

=== DATASET OVERVIEW ===
Total Songs in dataset: 899702

=== JUSTIN BIEBER SONGS ===
Total Justin Bieber songs found: 84
                                                          name     track_artists                                      album_name
                                      Friends (with BloodPop®)     Justin Bieber                        Friends (with BloodPop®)
                        Monster (Shawn Mendes & Justin Bieber)               NaN                                         Monster
                                    Confident - Single Version     Justin Bieber                                        Journals
                                     STAY (with Justin Bieber)     The Kid LAROI                       STAY (with Justin Bieber)
                       Juke Jam (feat. Justin Bieber & Towkio) Chance the Rapper                                   Coloring Book
                                attention (with Justin Bieber)               NaN                         

In [4]:
import sys; sys.path.append('..')

In [6]:
import difflib
import importlib
import pandas as pd
import numpy as np
import src.recommender

# Reload recommender module
importlib.reload(src.recommender)
from src.recommender import Song, UserProfile

print("🚀 Loading full dataset (899,000+ tracks)...")
df = pd.read_parquet('docs/tracks.parquet')

# Audio features for Cosine Similarity
audio_features = ['danceability', 'energy', 'valence', 'acousticness', 'speechiness', 'liveness']

# Prepare clean dataset
clean_df = df.dropna(subset=audio_features + ['name']).copy()

# Extract Release Year
clean_df['release_year'] = pd.to_datetime(clean_df['album_release_date'], errors='coerce').dt.year.fillna(2018)

# Smart artist name resolution
artist_display = (
    clean_df['track_artists']
    .fillna(clean_df['track_album_album'])
    .fillna(clean_df['album_name'])
    .fillna('Unknown Artist')
)
clean_df['artist_display'] = artist_display

def smart_search(query_text, max_results=15):
    q = query_text.strip().lower()
    
    # 1. Substring match
    mask_exact = (
        clean_df['name'].astype(str).str.lower().str.contains(q, na=False) |
        clean_df['artist_display'].astype(str).str.lower().str.contains(q, na=False)
    )
    matches = clean_df[mask_exact].drop_duplicates(subset=['name', 'artist_display']).head(max_results)
    
    # 2. Fuzzy match auto-correction for typos
    if len(matches) < 2:
        unique_artists = clean_df['artist_display'].dropna().unique()
        close_artist_matches = difflib.get_close_matches(query_text, [str(a) for a in unique_artists], n=5, cutoff=0.55)
        
        if close_artist_matches:
            print(f"💡 (Auto-corrected typo) Matching artist(s): {', '.join(close_artist_matches)}")
            fuzzy_mask = clean_df['artist_display'].astype(str).isin(close_artist_matches) | clean_df['name'].astype(str).isin(close_artist_matches)
            fuzzy_matches = clean_df[fuzzy_mask].drop_duplicates(subset=['name', 'artist_display']).head(max_results)
            matches = pd.concat([fuzzy_matches, matches]).drop_duplicates(subset=['name', 'artist_display']).head(max_results)
            
    return matches

def run_multi_signal_recommender():
    search_query = input("🔍 Enter a song title or artist name (e.g. 'Shawn Mendes', 'Justin Bieber', 'Taylor Swift'): ")
    if not search_query.strip():
        print("Empty search query. Exiting.")
        return

    matches = smart_search(search_query, max_results=15)
    
    if matches.empty:
        print(f"❌ No matching tracks found for '{search_query}'. Please check your spelling.")
        return

    print(f"\n🎵 Found {len(matches)} matching track(s):")
    for idx, (_, row) in enumerate(matches.iterrows(), 1):
        album = row.get('album_name', 'N/A')
        year = int(row['release_year']) if pd.notnull(row['release_year']) else 'N/A'
        print(f"  {idx}. '{row['name']}' — {row['artist_display']} (Year: {year} | Album: {album})")

    user_choice = input(f"\n👉 Select a song number (1-{len(matches)}) [Default: 1]: ")
    choice_str = user_choice.strip()
    choice_idx = int(choice_str) if choice_str and choice_str.isdigit() and 1 <= int(choice_str) <= len(matches) else 1

    selected_row = matches.iloc[choice_idx - 1]
    print(f"\n🎯 Selected Seed Track: '{selected_row['name']}' — {selected_row['artist_display']} ({int(selected_row['release_year'])})")
    print(f"   Genres: {selected_row.get('genres')}")
    print(f"   Audio Vector: Energy={selected_row['energy']:.2f}, Danceability={selected_row['danceability']:.2f}, Valence={selected_row['valence']:.2f}")

    seed_title = str(selected_row.get('name', '')).strip().lower()
    seed_artist = str(selected_row['artist_display']).lower()
    seed_genres = str(selected_row.get('genres', '')).lower()
    seed_year = float(selected_row['release_year'])

    # Candidates pool (exclude exact seed title)
    candidates_df = clean_df[clean_df['name'].astype(str).str.lower() != seed_title].copy()

    # =========================================================================
    # MULTI-SIGNAL SCORING MODEL
    # =========================================================================

    # SIGNAL 1: Audio Feature Cosine Similarity (35%)
    seed_audio_vec = selected_row[audio_features].values.astype(float)
    cand_audio_matrix = candidates_df[audio_features].values.astype(float)

    dot_products = np.dot(cand_audio_matrix, seed_audio_vec)
    norms = np.linalg.norm(cand_audio_matrix, axis=1) * np.linalg.norm(seed_audio_vec)
    norms[norms == 0] = 1e-9
    audio_sim = dot_products / norms  # Value between 0.0 and 1.0

    # SIGNAL 2: Genre Similarity (25%)
    c_genres = candidates_df['genres'].astype(str).str.lower()
    genre_sim = np.zeros(len(candidates_df))
    
    if 'pop' in seed_genres:
        is_pop = c_genres.str.contains('pop', na=False)
        is_rap = c_genres.str.contains('hip hop|rap|trap', regex=True, na=False)
        genre_sim[is_pop] = 0.8
        genre_sim[c_genres.str.contains('canadian pop|viral pop|dance pop|pop rock', regex=True, na=False)] = 1.0
        genre_sim[is_pop & is_rap] = 0.2
    else:
        base_g = seed_genres.split(',')[0].replace('[','').replace(']','').replace("'",'').strip()
        if base_g:
            genre_sim[c_genres.str.contains(base_g, na=False)] = 1.0

    # SIGNAL 3: Artist Similarity (20%)
    c_artists = candidates_df['artist_display'].astype(str).str.lower()
    pop_cluster = 'shawn mendes|justin bieber|charlie puth|ed sheeran|taylor swift|the chainsmokers|jonas brothers|lauv|one direction|post malone|tate mcrae|rihanna|the weeknd'
    
    artist_sim = np.where(
        c_artists == seed_artist, 
        1.0, 
        np.where(c_artists.str.contains(pop_cluster, regex=True, na=False), 0.7, 0.0)
    )

    # SIGNAL 4: Popularity Score (10%)
    pop_vals = candidates_df['artist_popularity'].fillna(candidates_df.get('popularity', 50)).fillna(50.0).astype(float).values / 100.0

    # SIGNAL 5: Release Year Proximity (10%)
    year_diffs = np.abs(candidates_df['release_year'].values - seed_year)
    year_sim = np.maximum(0.0, 1.0 - (year_diffs / 15.0))  # Decays smoothly over 15 years

    # FINAL COMPOSITE WEIGHTED SCORE (100% Total)
    final_score = (
        0.35 * audio_sim +
        0.25 * genre_sim +
        0.20 * artist_sim +
        0.10 * pop_vals +
        0.10 * year_sim
    )
    
    candidates_df['match_percentage'] = np.round(final_score * 100.0, 1)

    # Sort & deduplicate
    top_recs = candidates_df.sort_values(by='match_percentage', ascending=False).drop_duplicates(subset=['name']).head(20)

    print(f"\n=========================================================================")
    print(f"🌟 TOP 10 MULTI-SIGNAL RECOMMENDATIONS FOR: '{selected_row['name']}' ({selected_row['artist_display']})")
    print(f"=========================================================================")
    for i, (_, r) in enumerate(top_recs.iterrows(), 1):
        year_str = int(r['release_year']) if pd.notnull(r['release_year']) else 'N/A'
        print(f"{i}. {r['name']} — {r['artist_display']} (Match: {r['match_percentage']:.1f}%)")
        print(f"   Release Year: {year_str} | Genres: {r['genres']}")
        print(f"   Audio Profile: Energy={r['energy']:.2f}, Danceability={r['danceability']:.2f}, Valence={r['valence']:.2f}")
        print()

run_multi_signal_recommender()


🚀 Loading full dataset (899,000+ tracks)...

🎵 Found 15 matching track(s):
  1. 'Sunflower Feelings' — Sunflower Feelings (Year: 2017 | Album: Sunflower Feelings)
  2. 'Sunflower - Spider-Man: Into the Spider-Verse' — Spider-Man: Into the Spider-Verse (Soundtrack From & Inspired by the Motion Picture) (Year: 2018 | Album: Spider-Man: Into the Spider-Verse (Soundtrack From & Inspired by the Motion Picture))
  3. 'Sunflowers - Solo Piano' — Sunflowers (Year: 2021 | Album: Sunflowers)
  4. 'Sunflower' — Sunflower (Year: 2017 | Album: Sunflower)
  5. 'Sunflower' — The Sunflower Covers (From and Inspired by the Motion Picture "Spider-Man: Across the Spider-Verse") (Year: 2023 | Album: The Sunflower Covers (From and Inspired by the Motion Picture "Spider-Man: Across the Spider-Verse"))
  6. 'Sunflower, Vol. 6' — Harry Styles (Year: 2019 | Album: Fine Line)
  7. 'Rondo Of The House Of Sunflower (From Ponyo On The Cliff By The Sea) - Guitar' — Studio Ghibli Guitar Collection (Year: 2023 | Albu